# 01 — Data Ingest

Pulls 5 seasons of NFL data from nflverse (via `nflreadpy`) and caches to `data/raw/` as Parquet.

Run this **once at season start**, then re-run weekly to refresh the current season.

In [1]:
import nflreadpy as nfl
import polars as pl
from pathlib import Path

# Full nflverse history: pbp+schedules go to 1999, depth_charts to 2001, injuries to 2009.
SEASONS         = list(range(1999, 2026))
DEPTH_SEASONS   = list(range(2001, 2026))
INJURY_SEASONS  = list(range(2009, 2026))
DATA_DIR = Path('../data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Pulling seasons {SEASONS[0]}-{SEASONS[-1]} ({len(SEASONS)} seasons)')

Pulling seasons 1999-2025 (27 seasons)


## Schedules (results + closing lines)

In [2]:
schedules = nfl.load_schedules(seasons=SEASONS)
schedules.write_parquet(DATA_DIR / 'schedules.parquet')
print(f'Schedules rows: {schedules.shape[0]:,} | cols: {schedules.shape[1]}')
schedules.select(['game_id','season','week','home_team','away_team','home_score','away_score','spread_line','total_line']).head(10)

Schedules rows: 7,276 | cols: 46


game_id,season,week,home_team,away_team,home_score,away_score,spread_line,total_line
str,i32,i32,str,str,i32,i32,f64,f64
"""1999_01_MIN_ATL""",1999,1,"""ATL""","""MIN""",14,17,-4.0,49.0
"""1999_01_KC_CHI""",1999,1,"""CHI""","""KC""",20,17,-3.0,38.0
"""1999_01_PIT_CLE""",1999,1,"""CLE""","""PIT""",0,43,-6.0,37.0
"""1999_01_OAK_GB""",1999,1,"""GB""","""OAK""",28,24,9.0,43.0
"""1999_01_BUF_IND""",1999,1,"""IND""","""BUF""",31,14,-3.0,45.5
"""1999_01_SF_JAX""",1999,1,"""JAX""","""SF""",41,3,5.5,49.0
"""1999_01_CAR_NO""",1999,1,"""NO""","""CAR""",19,10,3.5,38.0
"""1999_01_NE_NYJ""",1999,1,"""NYJ""","""NE""",28,30,7.0,44.5
"""1999_01_ARI_PHI""",1999,1,"""PHI""","""ARI""",24,25,-3.0,37.0


## Play-by-play (EPA, success, all snaps)

In [3]:
# ~50k plays/season, this can take 1-2 min on first run (cached after)
pbp = nfl.load_pbp(seasons=SEASONS)
pbp.write_parquet(DATA_DIR / 'pbp.parquet')
print(f'PBP rows: {pbp.shape[0]:,} | cols: {pbp.shape[1]}')

PBP rows: 1,279,628 | cols: 372


## Team metadata

In [4]:
teams = nfl.load_teams()
teams.write_parquet(DATA_DIR / 'teams.parquet')
print(f'Teams: {teams.shape[0]}')
teams.select(['team_abbr','team_name','team_conf','team_division']).head()

Teams: 36


team_abbr,team_name,team_conf,team_division
str,str,str,str
"""ARI""","""Arizona Cardinals""","""NFC""","""NFC West"""
"""ATL""","""Atlanta Falcons""","""NFC""","""NFC South"""
"""BAL""","""Baltimore Ravens""","""AFC""","""AFC North"""
"""BUF""","""Buffalo Bills""","""AFC""","""AFC East"""
"""CAR""","""Carolina Panthers""","""NFC""","""NFC South"""


## Injuries (weekly status reports)

In [5]:
injuries = nfl.load_injuries(seasons=INJURY_SEASONS)
injuries.write_parquet(DATA_DIR / 'injuries.parquet')
print(f'Injury reports: {injuries.shape[0]:,}')
# Status values: Out, Doubtful, Questionable, Note, null
injuries.filter(pl.col('position') == 'QB').select(['season','week','team','full_name','report_status','report_primary_injury']).head(5)

Injury reports: 90,752


season,week,team,full_name,report_status,report_primary_injury
f64,f64,str,str,str,str
2009.0,1.0,"""ARI""","""Brian St. Pierre""","""Questionable""","""Back"""
2009.0,1.0,"""CIN""","""Carson Palmer""","""Probable""","""Ankle"""
2009.0,1.0,"""DAL""","""Stephen McGee""","""Questionable""","""Knee"""
2009.0,1.0,"""DEN""","""Kyle Orton""","""Questionable""","""right Finger"""
2009.0,1.0,"""DEN""","""Chris Simms""","""Questionable""","""Ankle"""


## Depth charts (weekly QB1/QB2 etc.)

In [6]:
depth = nfl.load_depth_charts(seasons=DEPTH_SEASONS)
depth.write_parquet(DATA_DIR / 'depth_charts.parquet')
print(f'Depth chart rows: {depth.shape[0]:,}')
depth.filter(pl.col('position') == 'QB').select(['season','week','club_code','full_name','depth_team']).head(5)

Depth chart rows: 1,423,400


season,week,club_code,full_name,depth_team
i32,i32,str,str,str
2001,5,"""ATL""","""Chris Chandler""","""1"""
2001,4,"""ATL""","""Chris Chandler""","""1"""
2001,3,"""ATL""","""Chris Chandler""","""1"""
2001,2,"""ATL""","""Chris Chandler""","""1"""
2001,6,"""ATL""","""Chris Chandler""","""1"""


## Done

Files written to `data/raw/`. Next: open `02_features.ipynb`.